In [13]:
%load_ext autoreload
%autoreload 2

import importlib 
m = importlib.import_module(".model", "src")
em = importlib.import_module(".em", "src")
from matplotlib import pyplot as plt
import numpy as np
import xgi
import scipy.special as ss
import cProfile

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
# generate some sample data

eta   = 0.5   # node retention in edges
gamma = 0.2   # poisson number of nodes from graph
beta  = 0.7   # poisson number of novel nodes

timesteps = int(1e4)

H = m.GrowingHypergraph()
H.add_edge((0, 1))
H.add_edge((2, 3))

for _ in range(timesteps):
    H.sample_edge(eta, gamma, beta, True)

In [15]:
# this appears to work as a form_arrays() type function for the E-step
# COULD BE PARALLELIZED RELATIVELY SIMPLY

edge_sizes = H.edge_size_sequence()
k_max = np.max(edge_sizes)
m = len(edge_sizes)
H.H.edges.members(1)

initial_edges = [0, 1]

all_nodes = set()

# num edges
# size of other edge e'
# size of intersection with e'
# number of nodes in (e - e') present in the hypergraph 

K = np.zeros((m, k_max, k_max, k_max), dtype = int)

for eid in H.edges:
    e = H.H.edges.members(eid)
    neighbor_edges = []
    for i in e: 
        neighbor_edges += [eid_ for eid_ in H.nodes.memberships(i) if eid_ < eid] 
        for eid_ in set(neighbor_edges): 
            f = H.H.edges.members(eid_)
            
            i = len(f)
            j = len(e.difference(f).intersection(all_nodes))
            k = len(e.intersection(f))
            
            K[eid, i-1, j-1, k-1] += 1
        
    all_nodes = all_nodes.union(e)
        
    # print(len(neighbor_edges))

### Comparison to the form_arrays() method of current EM implementation

In [16]:
# estimator likelihood definitions
# pmfs are used in E-step
# m_step estimators are the parameter estimators for each distribution
# used in m-step, using the matrix chi formed in the E-step

# edge sampling
def edge_sample_pmf(x, t, eta):
    return (eta**x)*((1-eta)**(t-x))

def edge_sample_m_step(x, t, chi):
    C = np.tril(chi,-1)
    return (x*C).sum() / (t*C).sum()

esl = em.EdgeSampleLikelihood(
    pmf = edge_sample_pmf, 
    m_step = edge_sample_m_step
    )

# edge sampling: guaranteed version
def guaranteed_edge_sample_pmf(x, t, eta):
    M = (eta**(x-1))*((1-eta)**(t-x)) # ?
    M[x == 0] = 0
    return M

def guaranteed_edge_sample_m_step(x, t, chi):
    C = np.tril(chi,-1)
    top = ((x-1)*C).sum()
    bottom = ((t-1)*C).sum()
    return top / bottom

eslg = em.EdgeSampleLikelihood(
    pmf = guaranteed_edge_sample_pmf, 
    m_step = guaranteed_edge_sample_m_step
    )

# addition of novel nodes not previously seen in the hypergraph
def novel_nodes_pmf(k, beta):
     return (beta**k)*np.exp(beta)/(ss.factorial(k))

def novel_nodes_m_step(k, chi):
    return k.mean()

nnl = em.NovelNodesLikelihood(pmf = novel_nodes_pmf, m_step = novel_nodes_m_step)

# addition of nodes from rest of hypergraph
def nodes_from_hypergraph_pmf(k, gamma):
    return (gamma**k)*np.exp(-gamma)/(ss.factorial(k))

def nodes_from_hypergraph_m_step(x, chi):
    return (np.tril(chi, -1)*x).sum(axis = 1).mean()

nhl = em.NodesFromHypergraphLikelihood(pmf = nodes_from_hypergraph_pmf, m_step = nodes_from_hypergraph_m_step)

model = em.EM(eslg, nhl, nnl) 

In [17]:
# maybe about twice as fast as the above
model.form_arrays(H)